# Skip-gram Softmax 概率计算 P(o|c)

> **CS224N 2025 Lecture 1 - WP03: Word2Vec / Skip-gram / Softmax**
>
> 本 notebook 演示 Skip-gram 模型中 softmax 如何把点积分数转化为概率分布。
> 公式：**P(o|c) = exp(u_o^T v_c) / sum_w exp(u_w^T v_c)**
>
> 官方锚点：Slides p27-30; Notes 3.2 Eq.4-5

## 这段代码在做什么

在 Skip-gram 模型中，每个词有**两套向量**：
- **v_w**：当词 w 作为**中心词 (center word)** 时
- **u_w**：当词 w 作为**上下文词 (context/outside word)** 时

给定一个中心词 c，模型要预测每个可能的上下文词 o 的概率。计算过程分三步：

1. **点积打分**：计算 u_w^T v_c（每个候选词和中心词的相似度）
2. **指数化**：exp(score)，使所有分数变为正数
3. **归一化**：除以所有 exp 之和（分母 = partition function），得到概率分布

这就是 **softmax** 函数——深度学习中极常用的「把任意实数分数变成概率」的工具。

## 1. 导入和设置

纯标准库实现，无需安装任何包，Colab 直接运行。

In [ ]:
import math
import json

print("=== Skip-gram Softmax P(o|c) ===")

## 2. 构建 mini vocabulary 和词向量

我们用一个 6 词的 mini vocabulary 来演示。每个词有两个向量（center v 和 context u），维度 d=3。

> **注意**：这里的向量值是确定性的伪随机值（用于教学演示），不是真正训练出来的。

In [ ]:
def make_deterministic_vectors(vocab, dim, seed=42):
    vectors = {}
    for i, word in enumerate(vocab):
        vec = []
        for d in range(dim):
            val = math.sin(seed * (i + 1) * 7.3 + (d + 1) * 13.7) * 2.0
            vec.append(round(val, 4))
        vectors[word] = vec
    return vectors

vocab = ["banking", "money", "crisis", "turning", "problems", "into"]
vocab_size = len(vocab)
dim = 3

V = make_deterministic_vectors(vocab, dim, seed=42)  # center vectors
U = make_deterministic_vectors(vocab, dim, seed=99)  # context vectors

print(f"vocab: {vocab}")
print(f"|V| = {vocab_size}, d = {dim}")
print()
for w in vocab:
    print(f"  v_{w} = {V[w]}")
print()
for w in vocab:
    print(f"  u_{w} = {U[w]}")

## 3. Softmax 函数

softmax 的核心操作：
1. **减去最大值**（max-subtraction trick，防止 exp 溢出）
2. **指数化** exp(x_i - max)
3. **归一化** 除以总和

> **为什么叫 "soft" max？** 因为它不像 argmax 只给最大值概率 1，而是给所有值分配一些概率。

In [ ]:
def dot_product(a, b):
    return sum(ai * bi for ai, bi in zip(a, b))

def softmax(scores):
    max_score = max(scores)
    exp_scores = [math.exp(s - max_score) for s in scores]
    sum_exp = sum(exp_scores)
    return [e / sum_exp for e in exp_scores]

print("softmax([2.0, 1.0, 0.1]) =", softmax([2.0, 1.0, 0.1]))

## 4. 计算 P(o|c) - Skip-gram softmax 核心

选择 "banking" 作为中心词，计算它预测每个候选上下文词的概率。

对应 Slides p28-30 和 Notes 3.2 Eq.4

In [ ]:
center_word = "banking"
print(f"center word c = '{center_word}'")
print(f"  v_c = {V[center_word]}")
print()

print("-- Step 1: dot products --")
scores = {}
for w in vocab:
    score = dot_product(U[w], V[center_word])
    scores[w] = round(score, 4)
    print(f"  u_{w}^T v_{center_word} = {score:.4f}")
print()

print("-- Step 2: exp(score) --")
exp_scores = {}
for w in vocab:
    exp_scores[w] = round(math.exp(scores[w]), 6)
    print(f"  exp({scores[w]:>8.4f}) = {exp_scores[w]:.6f}")
print()

sum_exp = sum(exp_scores.values())
print(f"-- Step 3: partition = {sum_exp:.6f} --")
print()

print("-- Step 4: P(o|c) --")
probs = {}
for w in vocab:
    probs[w] = round(exp_scores[w] / sum_exp, 6)
    print(f"  P({w:>10s} | {center_word}) = {probs[w]:.6f}")
print()
print(f"sum = {sum(probs.values()):.6f}")

## 输出怎么解释

**关键观察**：

1. **点积分数**：banking 自己得分最高 (0.4136)
2. **指数化后**：差距被放大 - exp(0.41) = 1.51 vs exp(-3.26) = 0.04
3. **归一化后**：banking 得 54%，into 只有 1.4%
4. **分母 = 2.799479**：需要对整个词汇表求和

**这就是为什么 full softmax 很贵**：真实 |V| = 50万+，每算一个 P(o|c) 都要加 50 万个 exp。WP04 的 negative sampling 就是为了解决这个问题。

## 5. 可视化概率分布

In [ ]:
try:
    import matplotlib.pyplot as plt
    ranked = sorted(probs.items(), key=lambda x: -x[1])
    words_r = [w for w, p in ranked]
    prob_vals = [p for w, p in ranked]
    fig, ax = plt.subplots(figsize=(8, 5))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#1a1a2e')
    colors = ['#e94560', '#f5a623', '#7ed321', '#4a90d9', '#9013fe', '#50e3c2']
    bars = ax.barh(range(len(words_r)), prob_vals, color=colors[:len(words_r)], height=0.6)
    ax.set_yticks(range(len(words_r)))
    ax.set_yticklabels(words_r, fontsize=12, color='#e0e0e0')
    ax.invert_yaxis()
    ax.set_xlabel('P(o | c)', color='#e0e0e0')
    ax.set_title('P(o | c="banking")', color='#e0e0e0')
    ax.tick_params(colors='#e0e0e0')
    for s in ['top','right']: ax.spines[s].set_visible(False)
    for s in ['bottom','left']: ax.spines[s].set_color('#444')
    for bar, prob in zip(bars, prob_vals):
        ax.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
                f'{prob:.4f}', va='center', color='#e0e0e0', fontsize=11)
    plt.tight_layout()
    plt.show()
except ImportError:
    ranked = sorted(probs.items(), key=lambda x: -x[1])
    for rank, (w, p) in enumerate(ranked, 1):
        print(f"  {rank}. {w:>10s}: {p:.6f}  {'#'*int(p*50)}")

## 和本讲哪个 waypoint 对应

本 notebook 对应 **WP03: Word2Vec / Skip-gram / Softmax**。

| 课堂概念 | 代码对应 |
|---------|---------|
| 每个词两个向量 (Slides p28) | V (center), U (context) |
| 点积 u_o^T v_c | dot_product(U[w], V[c]) |
| Softmax (Slides p30) | softmax() |
| Partition function | sum_exp = 2.799479 |
| P(o|c) (Notes Eq.4) | 概率表 |

## 容易误解的地方

1. **"banking 预测自己" 不是过拟合**：toy 向量恰好让 u_banking 和 v_banking 方向接近。真实训练取决于数据。

2. **两套向量不是两种含义**：只是角色不同。训练后通常只用 v (center) 作为最终词向量。

3. **softmax 分母是瓶颈**：|V|=6 只需加 6 个数；|V|=500,000 需要加 50 万个 exp。

4. **点积 != 余弦相似度**：这里用 raw dot product，不是归一化余弦。